# Cyclistic — Pandas Full Dataset

Same queries as the prototype but running on the full `data/processed/trips_clean.csv` (~14.8M rows, 2020–2022). This gives us real timings to compare against SQL and Spark.

In [2]:
import pandas as pd
import os
import time
import json
import psutil

notebook_start_time = time.time()
print("Notebook execution started...")

Notebook execution started...


## Data Import

Loading the full cleaned dataset. This already has bad rows removed from the pipeline so we just need to parse timestamps and compute ride length.

In [3]:
_t = time.time()

current_dir = os.getcwd()
data_path = os.path.join(current_dir, "..", "data", "processed", "trips_clean.csv")

df = pd.read_csv(data_path, parse_dates=["started_at", "ended_at"])

print(f"Rows loaded: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"Load time: {time.time() - _t:.2f}s")
df.head()

C:\Users\Adam McIntyre\AppData\Local\Temp\ipykernel_22264\2687366079.py:6: DtypeWarning: Columns (4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, parse_dates=["started_at", "ended_at"])


Rows loaded: 14,804,463
Columns: ['ride_id', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'member_casual']
Load time: 64.90s


,ride_id,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,member_casual
0,A847FADBBC638E45,2020-04-26 17:45:14,2020-04-26 18:12:03,Eckhart Park,86,Lincoln Ave & Diversey Pkwy,152.0,member
1,5405B80E996FF60D,2020-04-17 17:08:54,2020-04-17 17:17:03,Drake Ave & Fullerton Ave,503,Kosciuszko Park,499.0,member
2,5DD24A79A4E006F4,2020-04-01 17:54:13,2020-04-01 18:08:36,McClurg Ct & Erie St,142,Indiana Ave & Roosevelt Rd,255.0,member
3,2A59BBDF5CDBA725,2020-04-07 12:50:19,2020-04-07 13:02:31,California Ave & Division St,216,Wood St & Augusta Blvd,657.0,member
4,27AD306C119C6158,2020-04-18 10:22:59,2020-04-18 11:15:54,Rush St & Hubbard St,125,Sheridan Rd & Lawrence Ave,323.0,casual


## Compute Ride Length, Hour, and Day

In [4]:
_t = time.time()

df["ride_length_minutes"] = (
    (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60
)

# Drop same bad rows as the prototype filter
df = df[df["ride_length_minutes"].between(0, 1440, inclusive="neither")].copy()

df["hour"] = df["started_at"].dt.hour
df["day_of_week"] = df["started_at"].dt.day_name()

print(f"Rows after filter: {len(df):,}")
print(f"Prep time: {time.time() - _t:.2f}s")

Rows after filter: 14,779,921
Prep time: 6.47s


## Station-Based Insights

Top 10 start and end stations per user type, and average ride duration by start station.

In [5]:
_t = time.time()

for user_type in ["member", "casual"]:
    print(f"\nTop 10 start stations for {user_type}s:")
    top_start = (
        df[df["member_casual"] == user_type]["start_station_name"]
        .value_counts()
        .head(10)
    )
    print(top_start)

    print(f"\nTop 10 end stations for {user_type}s:")
    top_end = (
        df[df["member_casual"] == user_type]["end_station_name"]
        .value_counts()
        .head(10)
    )
    print(top_end)

avg_duration_station = (
    df.groupby(["start_station_name", "member_casual"])["ride_length_minutes"]
    .agg(["count", "mean"])
    .reset_index()
)
avg_duration_station = avg_duration_station[avg_duration_station["count"] >= 30]

print("\nAverage ride length (minutes) by start station and user type (>= 30 trips):")
print(avg_duration_station.sort_values("mean", ascending=False).head(20))
print(f"\nCell time: {time.time() - _t:.2f}s")


Top 10 start stations for members:
start_station_name
Clark St & Elm St               67158
Kingsbury St & Kinzie St        65084
Wells St & Concord Ln           60483
Wells St & Elm St               53852
Dearborn St & Erie St           51996
Broadway & Barry Ave            51286
St. Clair St & Erie St          51274
Wells St & Huron St             50316
Clinton St & Madison St         50193
Clinton St & Washington Blvd    46653
Name: count, dtype: int64

Top 10 end stations for members:
end_station_name
Clark St & Elm St               68326
Kingsbury St & Kinzie St        65008
Wells St & Concord Ln           62020
Dearborn St & Erie St           53395
Wells St & Elm St               53382
St. Clair St & Erie St          53149
Broadway & Barry Ave            52778
Clinton St & Madison St         51790
Clinton St & Washington Blvd    48754
Wells St & Huron St             48513
Name: count, dtype: int64

Top 10 start stations for casuals:
start_station_name
Streeter Dr & Grand Ave    

## Time of Day x Day of Week

Text heatmap showing trip counts by hour and day for each user type.

In [6]:
_t = time.time()

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
cat_type = pd.CategoricalDtype(categories=day_order, ordered=True)
df["day_of_week"] = df["day_of_week"].astype(cat_type)

for user_type in ["member", "casual"]:
    subset = df[df["member_casual"] == user_type]
    pivot = subset.pivot_table(
        index="day_of_week",
        columns="hour",
        values="ride_id",
        aggfunc="count",
        fill_value=0,
    )
    print(f"\nTrips table (day x hour) for {user_type}:")
    print(pivot)

    print(f"\nTop 5 day-hour cells for {user_type}:")
    top_cells = (
        pivot.stack()
        .sort_values(ascending=False)
        .head(5)
        .reset_index(name="trips")
    )
    for _, row in top_cells.iterrows():
        print(f"- {row['day_of_week']} @ {row['hour']}: {row['trips']:,} trips")

print(f"\nCell time: {time.time() - _t:.2f}s")


Trips table (day x hour) for member:
hour            0      1     2     3     4      5      6      7      8   \
day_of_week                                                               
Monday        6785   3801  2193  1623  2912  13477  38762  70854  79968   
Tuesday       5664   2833  1574  1232  3044  16085  47353  88358  96722   
Wednesday     6727   3222  1726  1209  2824  15534  46111  86224  96127   
Thursday      7815   3772  1977  1478  2754  14270  42985  81666  92682   
Friday       10641   5885  3224  1981  2890  12887  37569  65431  72911   
Saturday     20874  15343  8773  4643  3073   5135  13140  24610  40795   
Sunday       22552  15854  9559  5375  3608   4522  10484  17764  28511   

hour            9   ...     14     15      16      17      18     19     20  \
day_of_week         ...                                                       
Monday       48704  ...  60206  74056  106616  141453  113684  78551  50723   
Tuesday      54715  ...  58769  77701  118115  15

## Daily Trends

Trip counts by day of week and user type.

In [7]:
_t = time.time()

weekday_counts = (
    df.groupby(["day_of_week", "member_casual"])
    .size()
    .unstack(fill_value=0)
    .reindex(day_order)
)

print("Trips by day of week and user type (counts):")
print(weekday_counts)

print("\nDay-of-week share of trips by user type (row %):")
weekday_pct = weekday_counts.div(weekday_counts.sum(axis=1), axis=0).round(3)
print(weekday_pct)

print(f"\nCell time: {time.time() - _t:.2f}s")

Trips by day of week and user type (counts):
member_casual   casual   member
day_of_week                    
Monday          704022  1180546
Tuesday         674169  1298043
Wednesday       704321  1329205
Thursday        756411  1311182
Friday          898709  1237934
Saturday       1341736  1187196
Sunday         1123317  1033130

Day-of-week share of trips by user type (row %):
member_casual  casual  member
day_of_week                  
Monday          0.374   0.626
Tuesday         0.342   0.658
Wednesday       0.346   0.654
Thursday        0.366   0.634
Friday          0.421   0.579
Saturday        0.531   0.469
Sunday          0.521   0.479

Cell time: 1.42s


## Busy Stations

Top 10 stations ranked by total activity (departures + arrivals combined).

In [8]:
_t = time.time()

departures = df.groupby("start_station_name").size().rename("departures")
arrivals = df.groupby("end_station_name").size().rename("arrivals")

station_activity = (
    departures.to_frame()
    .join(arrivals.to_frame(), how="outer")
    .fillna(0)
)
station_activity["total_activity"] = (
    station_activity["departures"] + station_activity["arrivals"]
)

top_stations = station_activity.sort_values("total_activity", ascending=False).head(10)
print("Top 10 stations by total activity (departures + arrivals):")
print(top_stations)

print(f"\nCell time: {time.time() - _t:.2f}s")

Top 10 stations by total activity (departures + arrivals):
                          departures  arrivals  total_activity
Streeter Dr & Grand Ave     192903.0  196359.0        389262.0
Clark St & Elm St           107969.0  106860.0        214829.0
Michigan Ave & Oak St       105584.0  107120.0        212704.0
Wells St & Concord Ln       106075.0  106588.0        212663.0
Millennium Park             101190.0  103750.0        204940.0
Theater on the Lake          99074.0  100925.0        199999.0
Wells St & Elm St            91568.0   88729.0        180297.0
Kingsbury St & Kinzie St     89644.0   87043.0        176687.0
Broadway & Barry Ave         84281.0   85878.0        170159.0
Clark St & Armitage Ave      85566.0   83541.0        169107.0

Cell time: 3.04s


## Route Duration Groupby — Bottleneck Query

This is the query that hurts pandas on the full dataset. Grouping by every unique start and end station pair (50,000+ combinations) forces pandas to build a huge hash table in memory all at once. Compare this time to the Spark version.

In [9]:
_t = time.time()

route_duration = (
    df.dropna(subset=["start_station_name", "end_station_name"])
    .groupby(["start_station_name", "end_station_name"])["ride_length_minutes"]
    .agg(trip_count="count", avg_duration_minutes="mean")
    .reset_index()
)
route_duration = (
    route_duration[route_duration["trip_count"] >= 30]
    .sort_values("avg_duration_minutes", ascending=False)
    .reset_index(drop=True)
)
route_duration["avg_duration_minutes"] = route_duration["avg_duration_minutes"].round(2)

groupby_time = time.time() - _t
print(f"Groupby time: {groupby_time:.2f}s")
print(f"Unique routes with >= 30 trips: {len(route_duration)}")
print("\nTop 20 routes by avg duration (minutes):")
print(route_duration.head(20).to_string(index=False))

Groupby time: 6.60s
Unique routes with >= 30 trips: 58710

Top 20 routes by avg duration (minutes):
          start_station_name               end_station_name  trip_count  avg_duration_minutes
             Millennium Park      Clark St & Ida B Wells Dr          84                152.96
         Buckingham Fountain      Clark St & Ida B Wells Dr          43                137.50
       Prairie Ave & 43rd St          Calumet Ave & 51st St          36                135.66
    Fairbanks Ct & Grand Ave Museum of Science and Industry          35                125.55
      Benson Ave & Church St        Streeter Dr & Grand Ave          36                125.08
 Michigan Ave & Jackson Blvd      Clark St & Ida B Wells Dr          60                122.90
     Lake Park Ave & 53rd St     Stony Island Ave & 71st St          30                118.32
       Michigan Ave & Oak St      Chicago Ave & Sheridan Rd          30                117.95
        Yates Blvd & 75th St           Yates Blvd & 75

## Total Time and Memory

In [10]:
notebook_end_time = time.time()
total_seconds = notebook_end_time - notebook_start_time
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)

print("-" * 40)
print(f"Total Pandas Full Dataset Execution Time: {minutes} minutes and {seconds} seconds.")

process = psutil.Process(os.getpid())
memory_info = process.memory_info()
memory_usage_mb = memory_info.rss / (1024 * 1024)
print(f"Memory Usage: {memory_usage_mb:.2f} MB")
print("-" * 40)

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
results_dir = os.path.join(project_root, "results")
os.makedirs(results_dir, exist_ok=True)
metrics_file_path = os.path.join(results_dir, "metrics.json")

if os.path.exists(metrics_file_path):
    with open(metrics_file_path, "r") as f:
        try:
            metrics_data = json.load(f)
        except json.JSONDecodeError:
            metrics_data = {}
else:
    metrics_data = {}

metrics_data["pandas_full"] = {
    "total_execution_time_seconds": round(total_seconds, 2),
    "total_execution_time_formatted": f"{minutes}m {seconds}s",
    "memory_usage_mb": round(memory_usage_mb, 2)
}

with open(metrics_file_path, "w") as f:
    json.dump(metrics_data, f, indent=4)

print(f"Metrics updated in: {metrics_file_path}")

----------------------------------------
Total Pandas Full Dataset Execution Time: 2 minutes and 26 seconds.
Memory Usage: 4253.58 MB
----------------------------------------
Metrics updated in: c:\Users\Adam McIntyre\Cyclistic_Data\Cyclistic_Data\results\metrics.json
